# A Factory Is Not a Histogram

Executable companion to [the full industrial statistics article](article.md). This notebook works with **synthetic physical truths and measurements** so that we can separate process instability, measurement uncertainty, and specification mistakes. It runs the smaller deterministic experiment; publication evidence is generated with `python analysis.py`.

## 1. Process stability before capability

The first 70 rational subgroups are generated from a stationary Gaussian baseline. The physical mean shifts by +0.36 for the final 50 subgroups. The monitoring gauge adds a fixed +0.10 bias and iid repeatability error. The control limits are estimated **only from the known baseline** and held fixed. Even a truly stationary baseline can generate an occasional three-sigma signal; this would require investigation in practice.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from analysis import N_BASELINE, SUBGROUP_SIZE, LSL, USL, capability, crossed_anova_components, run_study

study = run_study(quick=True)
print('Baseline limits:', study.limits)
print('Known baseline Cpk:', study.capability_rows[0]['cpk'])
print('Descriptive mixed-phase index:', study.capability_rows[-1]['cpk'])
assert study.true_process.shape == (120, SUBGROUP_SIZE)

In [ ]:
xbars = study.measured_process.mean(axis=1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(xbars, color='tab:blue', label='Measured subgroup mean')
ax.axvline(N_BASELINE - .5, ls=':', color='black', label='Known shift')
ax.axhline(study.limits['xbar_ucl'], ls='--', color='tab:red', label='Frozen upper limit')
ax.axhline(study.limits['xbar_lcl'], ls='--', color='tab:red')
ax.set_xlabel('Subgroup')
ax.set_ylabel('Dimension')
ax.legend()
ax.grid(alpha=.2)
plt.show()

## 2. Crossed gauge R&R

Every one of twenty reference parts is measured by three operators, with three repetitions per operator–part cell. A balanced random-effects ANOVA separates repeatability, operator, and part×operator variances. The part variance reflects the selected reference parts, not the factory's within-process sigma. The gauge study **does not** identify absolute bias; that needs a traceable master.

In [ ]:
components = crossed_anova_components(study.gauge_design.values)
for name in ('repeatability', 'operator', 'part_operator', 'part'):
    print(name, 'estimated variance:', round(components[name], 5))
print('Gauge SD:', round(components['gauge_sd'], 3))
print('Estimated master bias:', round(study.bias_estimate, 3))
assert np.isclose(components['gauge_sd'], study.gauge_components['gauge_sd'])

## 3. Inspection decisions and false acceptance

A separate borderline inspection population supplies enough truly nonconforming parts for meaningful classification counts. We know latent dimensions only because this is a simulation. A false accept is an out-of-specification part that the gauge passes; a false reject is a conforming part that it fails. Figures and loss weights describe **per-10,000 produced-part** outcomes, which are different from conditional error rates within truly defective or conforming groups.

In [ ]:
for row in study.inspection_rows:
    print(row['reading_policy'], 'false accepts/10k:', round(float(row['false_accepts_per_10000']), 2), 'false rejects/10k:', round(float(row['false_rejects_per_10000']), 2))
guardbands = [float(row['guardband']) for row in study.guardband_rows]
loss = [float(row['illustrative_loss_per_10000']) for row in study.guardband_rows]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(guardbands, loss, '-o')
ax.set_xlabel('Inward guard band')
ax.set_ylabel('Illustrative weighted loss / 10k')
ax.grid(alpha=.2)
plt.show()

## 4. Change the system through a designed experiment

The two coded factors are feed speed and temperature. Six randomized replications of each of the four corner combinations are measured using five new physical parts per run, with four deliberately interspersed centerpoint runs. The interaction coefficient quantifies a **difference of effects**; treating the two factors as independent misses it. Risk projections assume the physical within-part standard deviation remains exactly 0.20 at all settings, which would need confirmation in a real plant.

In [ ]:
for row in study.factorial_effect_rows:
    print(row['term'], 'true coefficient:', round(float(row['true_beta']), 3), 'fitted:', round(float(row['estimated_beta']), 3), 'SE:', round(float(row['standard_error']), 3))
for row in study.factorial_setting_rows:
    print('speed:', row['speed_coded'], 'temp:', row['temperature_coded'], 'fitted nonconformance/10k:', round(10000 * float(row['fitted_normal_defect_rate']), 2))
assert len(study.factorial_setting_rows) == 4

## Reproduction and limitations

Use `python analysis.py` for the complete fixed-seed publication experiment and eight figures; `python analysis.py --quick` writes separately suffixed smoke outputs. These results describe a constructed factory with known Gaussian mechanisms, no missingness, no autocorrelated part dimensions, no operator switch in the time-order monitor, and no financial calibration of the loss weights. Do not transfer its capability or false-acceptance percentages to a real supplier without validating process stability and the entire measurement system. See the manuscript for source references.